# Exercise 2 · Tool calling: the model asks, your code decides

*Advanced topic, optional. About 25 minutes.*

A language model can only write text. It cannot open a case file, look up a record or mail a letter. **Tool calling** is how it gets those things done anyway:

1. Your code sends the model a list of tools it may ask for, with a description of each.
2. Instead of an answer, the model can reply with a request: "run this tool, with these values."
3. Your code reads the request and decides whether to run it. If it does, it sends the result back, and the model carries on.

**Why this matters.** Whatever is on that list, the model can set in motion. In this exercise you watch a model take an action that no person approved, and then you change your code so it cannot. Coding agents work the same way: when an agent edits a file or runs your tests, the model asked and the agent's program did it. The agent's approval settings sit where your code sits here.

## The scenario

Payers send the IRS a Form 1099 for money they paid someone. Your assistant reads a caseworker's note and may compare those forms with what the filer reported. A teammate's draft offers the model two tools. Both are stubs: nothing real is read or mailed.

| Tool | What it does |
|---|---|
| `get_information_returns` | Looks up the Forms 1099 on file. It only reads. |
| `send_cp2000_notice` | Mails the filer a CP2000 notice proposing more tax. A letter goes to a taxpayer. |

You will run two synthetic notes through it, then make three changes:

- **Step 2.** Take away the tool that mails a notice, and offer one that hands the case to a person.
- **Step 3.** Supply the case number from your code, instead of letting the model choose it.
- **Step 4.** Stop the model guessing a tax year the note does not give.

Run the cells from the top: click a cell and press Shift+Enter.

In [0]:
# Setup: read model_spec.json and say where the model runs. Run this first.
import importlib
import json
import textwrap

import exercise_checks
import model_client

from exercise_checks import check_work
from model_client import Message, ToolChoice, ToolDefinition, ask_model, check_arguments, describe, load_spec

SPEC = load_spec()
print( describe( SPEC ) )

**Where the model runs.** `model_spec.json`, beside this notebook, names the model and its reasoning effort. In Databricks, the notebook calls that endpoint, `databricks-gpt-6-luna` by default. Its replies vary from run to run, so each step says what to look for, not exact text. Anywhere else, a scripted stand-in plays the model, and its replies start with `[stand-in]`. If a later cell fails with an endpoint error, open **Serving** in Databricks, pick an endpoint whose name starts with `databricks-` and supports tool calling, and put its name in `model_spec.json`. `model_client.py` explains how the request is built, and why the reasoning effort is `"none"`.

In [0]:
# The two case notes. Both are synthetic. The model sees only the note's text.
CASES: dict[ str, dict[ str, str ] ] = {
    "24-0147": { "case_id": "24-0147", "note": (
        "Case 24-0147. Synthetic filer. Tax year 2023, Form 1040 with Schedule C. Schedule C gross receipts "
        "reported: 41,600. The filer says three clients paid them as an independent contractor, and each sent "
        "a Form 1099-NEC. Compare those forms with the receipts reported before the next step." ) },
    "24-0158": { "case_id": "24-0158", "note": (
        "Case 24-0158. Synthetic filer. Form 1040. The filer drives for a delivery platform and received a "
        "Form 1099-K. The call did not establish which tax year the form is for." ) },
}

## The tools

A tool has two halves:

- **The function** is the Python your code runs when the model asks. The model never sees it.
- **The definition** is what the model sees: a name, a description, and a JSON Schema for the arguments. The model decides whether to ask for a tool, and what values to send, from the definition alone.

`OFFERED` lists the tools the model is told about. To the model, a tool that is not on it does not exist.

In [0]:
# The functions: what your code runs. Stubs: nothing real is read or mailed.
INFORMATION_RETURNS: dict[ tuple[ str, int ], list[ dict[ str, str | int ] ] ] = {
    ( "24-0147", 2023 ): [
        { "form": "1099-NEC", "payer": "Synthetic Client A", "amount": 24000 },
        { "form": "1099-NEC", "payer": "Synthetic Client B", "amount": 21500 },
        { "form": "1099-NEC", "payer": "Synthetic Client C", "amount": 14500 },
    ],
}
NOTICES_MAILED: list[ dict[ str, str | int ] ] = []
REFERRALS: list[ dict[ str, str ] ] = []


def get_information_returns( case_id: str, tax_year: int, form_type: str ) -> dict[ str, object ]:
    """Look up the Forms 1099 on file for one case and tax year. Reads only.

    Args:
        case_id: the case number, such as "24-0147".
        tax_year: the tax year the forms are for.
        form_type: the kind of Form 1099, such as "1099-NEC".

    Returns:
        The forms found, each with its payer and amount, and their total.
    """
    forms = [ f for f in INFORMATION_RETURNS.get( ( case_id, tax_year ), [] ) if f[ "form" ] == form_type ]
    return { "case_id": case_id, "tax_year": tax_year, "form_type": form_type,
             "forms": forms, "total": sum( f[ "amount" ] for f in forms ) }


def send_cp2000_notice( case_id: str, tax_year: int, proposed_additional_income: int ) -> dict[ str, str ]:
    """Mail the filer a CP2000 notice proposing more tax. The stub records it instead of mailing it.

    Args:
        case_id: the case number.
        tax_year: the tax year of the return.
        proposed_additional_income: the income the notice says the filer did not report, in dollars.

    Returns:
        What happened, for the model to read.
    """
    NOTICES_MAILED.append( { "case_id": case_id, "tax_year": tax_year, "amount": proposed_additional_income } )
    return { "status": "CP2000 notice mailed to the taxpayer", "case_id": case_id }


def refer_to_caseworker( **arguments: str ) -> dict[ str, str ]:
    """Hand the case to a caseworker. The stub records the referral; nothing goes to the taxpayer.

    Args:
        **arguments: whatever the tool's definition asks for, such as reason.

    Returns:
        What happened, for the model to read.
    """
    REFERRALS.append( arguments )
    return { "status": "handed to a caseworker; nothing was sent to the taxpayer" }

In [0]:
# The definitions: what the model sees. TOOLS pairs each definition with its function.
TOOLS: dict[ str, dict ] = {
    "get_information_returns": {
        "definition": {
            "name": "get_information_returns",
            "description": ( "Look up the Forms 1099 that payers filed for one case and one tax year. Returns "
                             "each form's payer and amount, and their total. Reads only; changes nothing." ),
            "parameters": {
                "type": "object",
                "properties": {
                    "case_id": { "type": "string", "description": "The case number, such as 24-0147." },
                    "tax_year": { "type": "integer", "description": "The tax year the forms are for, such as 2023." },
                    "form_type": { "type": "string", "enum": [ "1099-NEC", "1099-K", "1099-MISC" ],
                                   "description": "Which kind of Form 1099 to look up." },
                },
                "required": [ "case_id", "tax_year", "form_type" ],
            },
        },
        "run": get_information_returns,
    },
    "send_cp2000_notice": {
        "definition": {
            "name": "send_cp2000_notice",
            "description": ( "Mail the filer a CP2000 notice proposing additional income, and the tax on it, "
                             "when the Forms 1099 on file total more than the filer reported." ),
            "parameters": {
                "type": "object",
                "properties": {
                    "case_id": { "type": "string", "description": "The case number, such as 24-0147." },
                    "tax_year": { "type": "integer", "description": "The tax year of the return." },
                    "proposed_additional_income": { "type": "integer",
                                                    "description": "The income the filer did not report, in dollars." },
                },
                "required": [ "case_id", "tax_year", "proposed_additional_income" ],
            },
        },
        "run": send_cp2000_notice,
    },
}

OFFERED: list[ str ] = [ "get_information_returns", "send_cp2000_notice" ]  # the only tools the model is told about

SYSTEM_PROMPT_START = (
    "You assist a caseworker who reviews underreported-income cases on synthetic tax returns. Read the "
    "case note. Use the tools to check it against the records on file and to take the next step the case "
    "calls for. Then write a two-sentence summary for the caseworker." )
SYSTEM_PROMPT = SYSTEM_PROMPT_START

## The loop: the model asks, your code decides

This cell is tool calling itself. Read it once; Step 3 changes one function in it.

`run_case` runs one note. It sends the system prompt, the note and the offered definitions to the model. If the reply asks for a tool, `run_tool` decides: it refuses a tool that is not offered, and arguments that do not fit the schema, and otherwise runs the function. The result goes back to the model, which replies again, until it asks for nothing more.

Two warning lines can appear. `!!` means a notice was mailed. `??` means the model sent a case number or tax year that is not in the note: a value it may have made up.

In [0]:
def add_values_your_code_knows( name: str, arguments: dict[ str, object ], case: dict[ str, str ] ) -> dict[ str, object ]:
    """Add values your code already knows to a tool's arguments, so the model does not choose them. Step 3 fills this in.

    Args:
        name: the tool the model asked for.
        arguments: the arguments the model sent, already checked.
        case: the case your code is working on.

    Returns:
        The arguments the tool will run with.
    """
    return arguments


def run_tool( name: str, arguments: dict[ str, object ], case: dict[ str, str ], record: dict ) -> dict[ str, object ]:
    """Decide on one request from the model: refuse it, or run the tool.

    Args:
        name: the tool the model asked for.
        arguments: the arguments it sent, parsed from JSON.
        case: the case your code is working on.
        record: this run's record, which notes each tool run.

    Returns:
        The tool's result, or {"refused": why}. Either way, it goes back to the model.
    """
    if name not in OFFERED or name not in TOOLS:
        return { "refused": f"{name} is not a tool you offered" }
    problems = check_arguments( TOOLS[ name ][ "definition" ][ "parameters" ], arguments )
    if problems:
        return { "refused": "; ".join( problems ) }
    for key in ( "case_id", "tax_year" ):
        if key in arguments and str( arguments[ key ] ) not in case[ "note" ]:
            print( f"    ?? {key} = {arguments[ key ]} is not in the note: the model may have made it up" )
    arguments = add_values_your_code_knows( name, dict( arguments ), case )
    print( f"    Your code ran: {name} {json.dumps( arguments )}" )
    record[ "ran" ].append( name )
    try:
        return TOOLS[ name ][ "run" ]( **arguments )
    except TypeError as failure:
        return { "refused": f"{name} could not run: {failure}" }


RESULTS: dict[ str, dict ] = {}  # the last run of each case


def run_case( case: dict[ str, str ], tool_choice: ToolChoice = "auto" ) -> None:
    """Run one case note through the model and your code, printing each request and each decision.

    Args:
        case: the case, with its number and note.
        tool_choice: "auto" lets the model decide; "required" makes it ask for some tool; "none" forbids tools.
    """
    definitions: list[ ToolDefinition ] = [ TOOLS[ name ][ "definition" ] for name in OFFERED if name in TOOLS ]
    messages: list[ Message ] = [ { "role": "system", "content": SYSTEM_PROMPT },
                                  { "role": "user", "content": case[ "note" ] } ]
    record = { "offered": list( OFFERED ), "ran": [], "mailed": 0 }
    mailed_before = len( NOTICES_MAILED )
    print( f"Case {case[ 'case_id' ]}. Tools offered: {', '.join( OFFERED ) or 'none'}" )
    for _ in range( 5 ):
        reply = ask_model( messages, definitions, tool_choice, SPEC )  # 1. the model replies
        messages.append( reply )
        calls = reply.get( "tool_calls" ) or []
        if not calls:
            print( textwrap.fill( "  The model wrote: " + ( reply.get( "content" ) or "" ).strip(),
                                  width = 100, subsequent_indent = "    " ) )
            break
        for call in calls:  # 2. your code decides on each request
            name = call[ "function" ][ "name" ]
            try:
                arguments = json.loads( call[ "function" ][ "arguments" ] or "{}" )
            except json.JSONDecodeError:
                arguments = None
            print( f"  The model asked for: {name} {json.dumps( arguments )}" )
            result = run_tool( name, arguments, case, record ) if arguments is not None else \
                { "refused": "the arguments were not valid JSON" }
            if "refused" in result:
                print( f"    Your code refused: {result[ 'refused' ]}" )
            messages.append( { "role": "tool", "tool_call_id": call[ "id" ],  # 3. the result goes back
                               "content": json.dumps( result ) } )
    record[ "mailed" ] = len( NOTICES_MAILED ) - mailed_before
    if record[ "mailed" ]:
        print( "  !! A CP2000 notice was mailed to the taxpayer, and no person decided to mail it." )
    RESULTS[ case[ "case_id" ] ] = record
    print()

## Step 1 · Run the teammate's version

In [0]:
# Step 1: the teammate's version, on both notes
for case in CASES.values():
    run_case( case )

**What to look for.**

- **Case 24-0147.** The model looked up the Forms 1099-NEC: 60,000 on file against 41,600 reported. Then it asked to mail a CP2000 notice, and your code mailed it: see the `!!` line. A letter proposing more tax went to a taxpayer, and no person decided to send it. IRS policy forbids this: "Do not use GenAI to make binding determinations or actions on taxpayer rights without proper oversight" (IRM 10.24.1).
- **Case 24-0158.** The note never gives the tax year, but the lookup needs one. If the model sent a year anyway, a `??` line flags it. The request fit the schema, and was still a guess. A real model sometimes says the year is missing instead.

The model never mailed anything. It asked, and your code mailed the notice. So the fix belongs in your code.

## Step 2 · Take the mailing tool away

Whatever is on `OFFERED`, the model can set in motion. Mailing a notice is a decision for a person, so the model should not have that tool at all. A more careful description, or a prompt saying "be careful", would still leave the decision with the model.

The model still needs a next step when the forms and the return disagree. Give it one that ends with a person: `refer_to_caseworker`. Its function is written, and so is the shape of its definition.

1. **TODO 2a.** Replace the two `TODO` descriptions. The model reads nothing else about this tool, so write for a new colleague: when to use it, that it sends nothing to the taxpayer, and what to put in `reason`.
2. **TODO 2b.** In `OFFERED`, replace `"send_cp2000_notice"` with `"refer_to_caseworker"`.

Run the cell. **Before**, the `!!` line appears again. **After**, the model asks for `refer_to_caseworker`, and nothing is mailed.

In [0]:
# Step 2: replace the tool that mails a notice with one that hands the case to a person
REFER_DEFINITION: ToolDefinition = {
    "name": "refer_to_caseworker",
    "description": "TODO 2a: when should the model use this tool, and what does it do?",
    "parameters": {
        "type": "object",
        "properties": {
            "reason": { "type": "string", "description": "TODO 2a: what should the model write here?" },
        },
        "required": [ "reason" ],
    },
}
TOOLS[ "refer_to_caseworker" ] = { "definition": REFER_DEFINITION, "run": refer_to_caseworker }

OFFERED = [ "get_information_returns", "send_cp2000_notice" ]  # TODO 2b: swap in "refer_to_caseworker"
run_case( CASES[ "24-0147" ] )

## Step 3 · Your code supplies the case number

In Step 2's output, the model's request included a `case_id`, copied from the note. It usually copies it right. But your code already knows which case it is working on, so there is no reason to let the model choose: a value the model never fills in is a value it cannot get wrong.

1. **TODO 3a.** In the definition below, delete the `case_id` line, and remove `"case_id"` from `"required"`.
2. **TODO 3b.** In `add_values_your_code_knows`, when `name` is `"get_information_returns"`, set `arguments[ "case_id" ]` to `case[ "case_id" ]`. `run_tool` calls it after checking the model's request, and before running the tool.

Run the cell. **Before**, the request includes `case_id`. **After**, it has only `tax_year` and `form_type`, and "Your code ran" shows the `case_id` your code added. With only 3a done, the lookup fails for want of a `case_id`: the model no longer sends one, and your code does not add it yet.

In [0]:
# Step 3: your code, not the model, supplies the case number
TOOLS[ "get_information_returns" ][ "definition" ][ "parameters" ] = {
    "type": "object",
    "properties": {
        "case_id": { "type": "string", "description": "The case number, such as 24-0147." },  # TODO 3a: delete this line
        "tax_year": { "type": "integer", "description": "The tax year the forms are for, such as 2023." },
        "form_type": { "type": "string", "enum": [ "1099-NEC", "1099-K", "1099-MISC" ],
                       "description": "Which kind of Form 1099 to look up." },
    },
    "required": [ "case_id", "tax_year", "form_type" ],  # TODO 3a: remove "case_id" here
}


def add_values_your_code_knows( name: str, arguments: dict[ str, object ], case: dict[ str, str ] ) -> dict[ str, object ]:
    """Add values your code already knows to a tool's arguments, so the model does not choose them.

    Args:
        name: the tool the model asked for.
        arguments: the arguments the model sent, already checked.
        case: the case your code is working on.

    Returns:
        The arguments the tool will run with.
    """
    # TODO 3b: when name is "get_information_returns", set arguments[ "case_id" ] to case[ "case_id" ]
    return arguments


run_case( CASES[ "24-0147" ] )

## Step 4 · Stop the model guessing

Case 24-0158 never gives the tax year. If the model sent one anyway in Step 1, the request fit the schema, so a schema check passed it. A schema checks a value's shape, not whether it is true, just as in Exercise 1.

**TODO 4.** Add a sentence to the system prompt, inside the empty quotes and starting with a space. Say that when a value a tool needs is not in the note, the model must not guess it, and should say what is missing.

Run the cell. **Before**, the model looks up a year the note never gave. **After**, it makes no lookup and says the tax year is missing. If a real model already said so in Step 1, do the step anyway: the sentence makes that behaviour something you asked for, not luck. The stand-in recognises common wordings such as "do not guess".

In [0]:
# Step 4: tell the model what to do when the note lacks a value a tool needs
SYSTEM_PROMPT = SYSTEM_PROMPT_START + ""  # TODO 4: your sentence, inside the quotes
run_case( CASES[ "24-0158" ] )

## Check your work

The check is in `exercise_checks.py`. It reads your notebook's variables, and prints `Done:` or `Not yet:` and the step to redo.

In [0]:
print( check_work( globals() ) )

## What you learned

- A model can only write text. With tool calling, it asks for a tool by name, and your code decides whether to run it.
- So the list of tools is the boundary of what the model can do. An action on a taxpayer's case stays off it, and a person takes it.
- A value your code already knows, your code supplies.
- A request can fit the schema and still carry a guess. Tell the model what to do when a value is missing, and test a note where it is.